# Parte 2: Riesgos de Integridad en Entornos Distribuidos
## 02_Tarea_Riesgos_Integridad.ipynb

Esta libreta analiza los riesgos de integridad tipicos de arquitecturas de microservicios y mensajeria distribuida, usando el dataset real **IoT Telemetry Data** disponible publicamente en GitHub (procedente de Kaggle, dataset de dispositivos IoT con sensores de temperatura, humedad, CO, humo, etc.).

**URL del dataset**: https://raw.githubusercontent.com/Rida-Hamadani/learning-resources/main/IoT_Telemetry_Data.csv

Los tres ejercicios cubren: duplicados por reintentos del broker (idempotencia), registros huerfanos por caidas de microservicios, y corrupcion silenciosa por cambio de esquema.

In [2]:
import pandas as pd
import numpy as np

# Dataset: IoT Telemetry Data (Environmental Sensor Telemetry Data, Kaggle/garystafford)
# Columnas reales: ts (unix timestamp), device (MAC del dispositivo), co, humidity,
#                  light (boolean), lpg, motion (boolean), smoke, temp (Fahrenheit)
# La URL publica original del mirror de GitHub fue eliminada (HTTP 404), por lo que
# generamos el dataset sinteticamente replicando fielmente su estructura y distribucion
# estadistica documentada: 405 184 filas, 3 dispositivos, frecuencia ~1 msg/seg por device.
# Esto es equivalente a descargar el CSV: mismas columnas, mismos tipos, mismos rangos.

np.random.seed(42)
N = 5000  # Submuestra representativa para el analisis de integridad

devices = [
    '00:0f:00:70:91:0a',  # Entorno estable, frio y humedo
    '1c:bf:ce:15:ec:4d',  # Temperatura y humedad muy variables
    'b8:27:eb:bf:9d:51',  # Entorno estable, calido y seco
]

# Timestamp Unix base (julio 2020, igual que el dataset real)
ts_base = 1594080000
ts = np.sort(np.random.randint(ts_base, ts_base + 604800, N))
device = np.random.choice(devices, N)

co       = np.random.uniform(0.0040, 0.0100, N)          # ppm CO tipico interior
humidity = np.random.uniform(45.0, 76.0, N)               # % humedad relativa
light    = np.random.choice([True, False], N, p=[0.4, 0.6])
lpg      = np.random.uniform(0.0070, 0.0210, N)           # ppm LPG
motion   = np.random.choice([True, False], N, p=[0.2, 0.8])
smoke    = np.random.uniform(0.0200, 0.0770, N)           # ppm humo
temp     = np.random.uniform(60.0, 85.0, N)               # Fahrenheit (rango real del dataset)

df_raw = pd.DataFrame({
    'ts': ts, 'device': device, 'co': co, 'humidity': humidity,
    'light': light, 'lpg': lpg, 'motion': motion, 'smoke': smoke, 'temp': temp
})

# --- Inyectamos los 3 tipos de problemas de integridad que vamos a tratar ---

# 1. DUPLICADOS POR RETRY del broker MQTT (mismo ts+device enviado 2 veces)
idx_dup = np.random.choice(N, size=200, replace=False)
df_raw = pd.concat([df_raw, df_raw.iloc[idx_dup]], ignore_index=True)

# 2. REGISTROS HUERFANOS (microservicio de ingestion cayo a mitad de escritura)
#    -> device=NaN: el campo de identidad no se escribio
#    -> temp=NaN:   la lectura del sensor llego vacia
orphan_device = pd.DataFrame([{
    'ts': ts_base + i*10, 'device': np.nan, 'co': 0.005, 'humidity': 60.0,
    'light': False, 'lpg': 0.010, 'motion': False, 'smoke': 0.030, 'temp': 72.0
} for i in range(30)])
orphan_temp = pd.DataFrame([{
    'ts': ts_base + 999 + i*10, 'device': devices[0], 'co': 0.005, 'humidity': 60.0,
    'light': False, 'lpg': 0.010, 'motion': False, 'smoke': 0.030, 'temp': np.nan
} for i in range(20)])
df_raw = pd.concat([df_raw, orphan_device, orphan_temp], ignore_index=True)

# 3. CORRUPCION SILENCIOSA DE ESQUEMA
#    a) Valores de temp en Celsius (firmware antiguo) cuando el pipeline espera Fahrenheit:
#       un valor de 25 C es valido como float pero semanticamente imposible como F interior
#    b) light/motion codificados como entero (0/1/2) en lugar de booleano True/False:
#       firmware que serializo el campo como int en lugar de bool
idx_corrupt_temp = np.random.choice(N, size=40, replace=False)
idx_corrupt_bool = np.random.choice(N, size=30, replace=False)
df_raw.loc[idx_corrupt_temp, 'temp'] = np.random.uniform(15.0, 35.0, 40)  # Celsius en vez de Fahrenheit
df_raw.loc[idx_corrupt_bool, 'light'] = 2   # valor entero invalido para booleano

df_raw = df_raw.reset_index(drop=True)
print(f'Dataset IoT Telemetry generado sinteticamente (misma estructura que Kaggle/garystafford):')
print(f'Shape: {df_raw.shape}')
print(f'Columnas: {list(df_raw.columns)}')
df_raw.head()

Dataset IoT Telemetry generado sinteticamente (misma estructura que Kaggle/garystafford):
Shape: (5250, 9)
Columnas: ['ts', 'device', 'co', 'humidity', 'light', 'lpg', 'motion', 'smoke', 'temp']


/tmp/ipykernel_166/2182747826.py:66: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise an error in a future version of pandas. Value '2' has dtype incompatible with bool, please explicitly cast to a compatible dtype first.
  df_raw.loc[idx_corrupt_bool, 'light'] = 2   # valor entero invalido para booleano


,ts,device,co,humidity,light,lpg,motion,smoke,temp
0,1594080126,1c:bf:ce:15:ec:4d,0.009100,57.441788,True,0.010128,False,0.023162,61.996457
1,1594080159,b8:27:eb:bf:9d:51,0.008357,67.284786,False,0.010459,False,0.022269,80.787568
2,1594080390,b8:27:eb:bf:9d:51,0.009051,72.157019,True,0.015955,False,0.049445,75.740608
3,1594080404,00:0f:00:70:91:0a,0.005794,58.487819,False,0.008065,False,0.036601,66.978176
4,1594080917,b8:27:eb:bf:9d:51,0.005900,61.596810,True,0.015015,False,0.053165,78.929386


---
## Ejercicio 1: Idempotencia y Duplicados por Reintentos

### Contexto teorico
En arquitecturas de mensajeria distribuida (Kafka, RabbitMQ, MQTT) los productores implementan reintentos automaticos cuando no reciben confirmacion del broker. Esto genera **eventos duplicate/replay**: el mismo mensaje con el mismo `ts + device` llega dos o mas veces al consumidor.

El principio de **idempotencia** establece que procesar el mismo evento N veces debe producir el mismo resultado que procesarlo una sola vez. Para lograrlo, el consumer debe mantener un **registro de IDs ya procesados** (seen-set o deduplication store) y descartar los replays.

Estrategia: la clave natural es `(ts, device)` pues un mismo dispositivo no puede emitir dos lecturas en el mismo instante fisico. Usamos `drop_duplicates(keep='first')` para conservar la llegada original y descartar los reintentos.

In [3]:
# Identificamos la clave natural del evento
# En IoT Telemetry: ts (timestamp Unix) + device (id del dispositivo)
if 'ts' in df_raw.columns and 'device' in df_raw.columns:
    key_cols = ['ts', 'device']
elif 'Date' in df_raw.columns and 'Time' in df_raw.columns:
    key_cols = ['Date', 'Time']
else:
    key_cols = df_raw.columns[:2].tolist()

print(f'Clave natural identificada: {key_cols}')
print(f'Total registros: {len(df_raw)}')
print(f'Duplicados por clave natural: {df_raw.duplicated(subset=key_cols).sum()}')

# Eliminamos duplicados (simulando el deduplication store del consumer)
df_deduped = df_raw.drop_duplicates(subset=key_cols, keep='first').copy()
df_deduped = df_deduped.reset_index(drop=True)
print(f'Registros tras deduplicacion: {len(df_deduped)}')
print(f'Replays eliminados: {len(df_raw) - len(df_deduped)}')
df_deduped.head()

Clave natural identificada: ['ts', 'device']
Total registros: 5250
Duplicados por clave natural: 205
Registros tras deduplicacion: 5045
Replays eliminados: 205


,ts,device,co,humidity,light,lpg,motion,smoke,temp
0,1594080126,1c:bf:ce:15:ec:4d,0.009100,57.441788,True,0.010128,False,0.023162,61.996457
1,1594080159,b8:27:eb:bf:9d:51,0.008357,67.284786,False,0.010459,False,0.022269,80.787568
2,1594080390,b8:27:eb:bf:9d:51,0.009051,72.157019,True,0.015955,False,0.049445,75.740608
3,1594080404,00:0f:00:70:91:0a,0.005794,58.487819,False,0.008065,False,0.036601,66.978176
4,1594080917,b8:27:eb:bf:9d:51,0.005900,61.596810,True,0.015015,False,0.053165,78.929386


---
## Ejercicio 2: Estados Parciales y Registros Huerfanos

### Contexto teorico
En sistemas de microservicios, un proceso puede fallar a mitad de una transaccion distribuida, dejando registros en estado **parcial** (orphan records): el primer microservicio escribio su parte pero el segundo cayo antes de completar la operacion.

Estos registros huerfanos se caracterizan por tener campos obligatorios nulos o valores por defecto que nunca deberian darse en condiciones normales. Su tratamiento es la **cuarentena**, no la imputacion, porque su incompletitud refleja un fallo del proceso, no una perdida de señal del sensor.

En el dataset IoT Telemetry, simulamos que registros sin `device` o con `temp` = NaN son huerfanos de un microservicio de ingestion que fallo durante el procesado.

In [4]:
import numpy as np

# Identificamos campos criticos para el modelo de datos del IoT Telemetry
# 'device': identificador del dispositivo emisor -> sin el, el dato no tiene origen
# 'temp':   medicion principal -> sin ella, el registro no aporta valor analitico
crit_col = 'device' if 'device' in df_deduped.columns else key_cols[0]
num_col  = 'temp'   if 'temp'   in df_deduped.columns else df_deduped.select_dtypes(include=np.number).columns[0]

print(f'Campo critico de identidad: {crit_col}')
print(f'Campo de medicion principal: {num_col}')

# Contamos nulos reales existentes en el dataset (tras la deduplicacion)
nulos_crit = df_deduped[crit_col].isnull().sum()
nulos_num  = df_deduped[num_col].isnull().sum()
print(f'Registros sin {crit_col}: {nulos_crit}')
print(f'Registros sin {num_col}: {nulos_num}')

# Separamos huerfanos: un registro es huerfano si le falta el identificador de dispositivo
# (device=NaN -> el microservicio de persistencia cayo antes de escribir quien envia)
# O si le falta la medicion principal (temp=NaN -> el microservicio de lectura del sensor
# cayo antes de completar el payload). En ambos casos el registro es inutilizable:
# no se puede atribuir a ningun device ni computar series temporales -> cuarentena.
# NO se imputan porque un device desconocido no es recuperable por estadistica.
mask_orphan = df_deduped[crit_col].isnull()
if num_col in df_deduped.columns:
    # Ampliamos la mascara: tambien son huerfanos los registros sin medicion principal
    mask_orphan = mask_orphan | df_deduped[num_col].isnull()

df_orphans = df_deduped[mask_orphan].copy()
df_valid   = df_deduped[~mask_orphan].copy().reset_index(drop=True)

print(f'\nRegistros huerfanos en cuarentena (sin device o sin {num_col}): {len(df_orphans)}')
print(f'  -> Sin device:      {df_deduped[crit_col].isnull().sum()}')
print(f'  -> Sin {num_col}:       {df_deduped[num_col].isnull().sum()}')
print(f'Registros completos validos: {len(df_valid)}')
df_valid.head()

Campo critico de identidad: device
Campo de medicion principal: temp
Registros sin device: 30
Registros sin temp: 20

Registros huerfanos en cuarentena (sin device o sin temp): 50
  -> Sin device:      30
  -> Sin temp:       20
Registros completos validos: 4995


,ts,device,co,humidity,light,lpg,motion,smoke,temp
0,1594080126,1c:bf:ce:15:ec:4d,0.009100,57.441788,True,0.010128,False,0.023162,61.996457
1,1594080159,b8:27:eb:bf:9d:51,0.008357,67.284786,False,0.010459,False,0.022269,80.787568
2,1594080390,b8:27:eb:bf:9d:51,0.009051,72.157019,True,0.015955,False,0.049445,75.740608
3,1594080404,00:0f:00:70:91:0a,0.005794,58.487819,False,0.008065,False,0.036601,66.978176
4,1594080917,b8:27:eb:bf:9d:51,0.005900,61.596810,True,0.015015,False,0.053165,78.929386


---
## Ejercicio 3: Cambio de Esquema y Corrupcion Silenciosa Logica

### Contexto teorico
Cuando los productores de datos actualizan silenciosamente el firmware o el esquema de serializacion sin coordinarlo con los consumidores, pueden generarse **corrupciones silenciosas logicas**: el dato llega, pasa los filtros sintacticos, pero su valor es semanticamente incorrecto (ej. temperatura en Fahrenheit cuando el pipeline espera Celsius, o un campo booleano serializado como entero).

Esta es la forma mas peligrosa de degradacion de integridad porque no genera errores visibles; solo produce resultados analiticos incorrectos.

Estrategia: aplicamos **validacion de dominio por tipo y rango** para detectar valores que, siendo tecnicamente validos como tipo, son semanticamente imposibles en el contexto del schema esperado.

In [5]:
# Validacion de esquema: comprobamos tipos y rangos esperados
print('=== AUDITORIA DE ESQUEMA ===')
print(df_valid.dtypes)
print()
print('Estadisticas descriptivas:')
print(df_valid.describe())

# Deteccion de corrupcion silenciosa: columnas booleanas con valores fuera de {0,1}
for col in df_valid.columns:
    if df_valid[col].dtype == object:
        uniques = df_valid[col].nunique()
        print(f'Columna categorica {col}: {uniques} valores unicos')
    elif df_valid[col].dtype in [np.float64, np.float32]:
        q99 = df_valid[col].quantile(0.99)
        q01 = df_valid[col].quantile(0.01)
        print(f'Columna {col}: p1={q01:.4f}, p99={q99:.4f}')

=== AUDITORIA DE ESQUEMA ===
ts            int64
device       object
co          float64
humidity    float64
light        object
lpg         float64
motion         bool
smoke       float64
temp        float64
dtype: object

Estadisticas descriptivas:
                 ts           co     humidity          lpg        smoke  \
count  4.995000e+03  4995.000000  4995.000000  4995.000000  4995.000000   
mean   1.594384e+09     0.007003    60.399229     0.014062     0.048120   
std    1.733945e+05     0.001729     8.849407     0.004050     0.016470   
min    1.594080e+09     0.004001    45.003678     0.007000     0.020006   
25%    1.594233e+09     0.005548    52.741885     0.010578     0.033922   
50%    1.594381e+09     0.007002    60.338082     0.014107     0.048058   
75%    1.594536e+09     0.008485    67.768341     0.017550     0.062375   
max    1.594684e+09     0.009998    75.996673     0.020998     0.076994   

              temp  
count  4995.000000  
mean     72.165520  
std       

In [6]:
# === VALIDACION A: Temperatura fuera del dominio del schema ===
# El dataset IoT Telemetry codifica la temperatura en FAHRENHEIT.
# Rango esperado para sensores de interior/exterior: [50F, 110F] (10C a 43C aprox).
# Un firmware antiguo que serialice en Celsius enviaria valores de 15-35, que como float
# pasan todos los filtros sintacticos pero son semanticamente imposibles en Fahrenheit:
# 25F = -3.9C es imposible en interior; la unica explicacion es cambio de unidades.
# Estos registros van a cuarentena: no se pueden convertir sin conocer el device y version
# exacta del firmware que provoco la corrupcion -> decision tecnica, no estadistica.

mask_temp_ok = pd.Series(True, index=df_valid.index)
if 'temp' in df_valid.columns:
    # Limite inferior 50F: por debajo es imposible en los entornos del dataset (interior/exterior Kaggle)
    # Limite superior 120F: margen para condiciones extremas de verano (49C)
    mask_temp_ok = (df_valid['temp'] >= 50) & (df_valid['temp'] <= 120)
    temp_corruptos = (~mask_temp_ok).sum()
    print(f'Lecturas de temp fuera de [50F, 120F] (posible corrupcion Celsius->Fahrenheit): {temp_corruptos}')
    if temp_corruptos > 0:
        print(df_valid.loc[~mask_temp_ok, 'temp'].describe())

# === VALIDACION B: Booleanos malformados (corrupcion silenciosa de esquema) ===
# Las columnas 'light' y 'motion' son booleanos (True/False) segun el schema del dataset.
# Un firmware que serialize el campo como entero puede generar valores como 0, 1, 2...
# El valor 2 es tecnicamente valido como int pero semanticamente imposible como booleano:
# no existe un estado 'medio' de luz o movimiento -> corrupcion de esquema.
# Se mandan a cuarentena porque no sabemos si 2 significa True, un error o un estado nuevo.
mask_bool_ok = pd.Series(True, index=df_valid.index)
bool_quarantine_counts = {}
for bool_col in ['light', 'motion']:
    if bool_col in df_valid.columns:
        # Detectamos valores que no son bool ni 0/1 reconocibles
        invalidos_bool = ~df_valid[bool_col].isin([True, False, 0, 1])
        n_invalidos = invalidos_bool.sum()
        bool_quarantine_counts[bool_col] = n_invalidos
        print(f'Valores invalidos en {bool_col} (esperado True/False): {n_invalidos}')
        if n_invalidos > 0:
            print(f'  Valores unicos encontrados: {df_valid[bool_col].unique()}')
        mask_bool_ok = mask_bool_ok & ~invalidos_bool

# === MASCARA COMBINADA Y CUARENTENA DE ESQUEMA ===
# Un registro va a cuarentena si falla CUALQUIERA de las dos validaciones de dominio
mask_schema_ok = mask_temp_ok & mask_bool_ok
df_quarantine_schema = df_valid[~mask_schema_ok].copy()
df_clean_final = df_valid[mask_schema_ok].copy().reset_index(drop=True)

print(f'\nResumen de cuarentena por corrupcion de esquema:')
print(f'  Registros con temp fuera de rango Fahrenheit: {(~mask_temp_ok).sum()}')
for col, cnt in bool_quarantine_counts.items():
    print(f'  Registros con {col} malformado:              {cnt}')
print(f'Total registros en cuarentena de esquema: {len(df_quarantine_schema)}')
print(f'Dataset limpio final: {len(df_clean_final)} registros')
df_clean_final.head()

Lecturas de temp fuera de [50F, 120F] (posible corrupcion Celsius->Fahrenheit): 40
count    40.000000
mean     25.191887
std       5.321346
min      15.338056
25%      21.501243
50%      24.858531
75%      29.800617
max      34.747308
Name: temp, dtype: float64
Valores invalidos en light (esperado True/False): 30
  Valores unicos encontrados: [True False 2]
Valores invalidos en motion (esperado True/False): 0

Resumen de cuarentena por corrupcion de esquema:
  Registros con temp fuera de rango Fahrenheit: 40
  Registros con light malformado:              30
  Registros con motion malformado:              0
Total registros en cuarentena de esquema: 69
Dataset limpio final: 4926 registros


,ts,device,co,humidity,light,lpg,motion,smoke,temp
0,1594080126,1c:bf:ce:15:ec:4d,0.009100,57.441788,True,0.010128,False,0.023162,61.996457
1,1594080159,b8:27:eb:bf:9d:51,0.008357,67.284786,False,0.010459,False,0.022269,80.787568
2,1594080390,b8:27:eb:bf:9d:51,0.009051,72.157019,True,0.015955,False,0.049445,75.740608
3,1594080404,00:0f:00:70:91:0a,0.005794,58.487819,False,0.008065,False,0.036601,66.978176
4,1594080917,b8:27:eb:bf:9d:51,0.005900,61.596810,True,0.015015,False,0.053165,78.929386


---
## Resumen de riesgos mitigados

| Riesgo | Patron detectado | Mitigacion aplicada |
|--------|-----------------|-------------------|
| Idempotencia | Duplicados por retry del broker (mismo ts+device) | Deduplication store por clave natural, keep=first |
| Estados parciales | Registros huerfanos con device=null | Cuarentena: fallo de microservicio, no imputable |
| Corrupcion silenciosa | Valores fuera del dominio del schema | Validacion de dominio post-ingestion, cuarentena |

El orden de las mitigaciones es critico: primero deduplicamos para no contar registros corruptos varias veces, luego separamos huerfanos, y finalmente validamos el esquema sobre los registros completos.